# 5 Gate, highway, or route: one idea from three papers

**This is not a time-series notebook.** The gate/highway/MoE family composes with any
`(batch, hidden)` tensor — images, text, tabular data, anything. It sits here because
this lesson already has a window and a baseline table to beat, the same reason
`1_pytorch_intro/04_search` introduced ray.tune where a two-second training run made it
obviously worth the investment rather than because tuning is somehow a time-series idea.


## 5.1 One idea, three resolutions

| component | mechanism | where it comes from |
|---|---|---|
| `Gate` | `sigmoid(Wx) * x` -- a learned per-dimension scale | the gate inside every LSTM cell (Hochreiter & Schmidhuber, 1997) and the reset/update gates of the GRU (Cho et al., 2014, *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*) |
| `Highway` | `g * T(x) + (1 - g) * x` -- a learned interpolation between a transform and the identity | Highway Networks (Srivastava, Greff & Schmidhuber, 2015), built explicitly to let very deep plain networks train at all -- the direct ancestor of the ResNet residual connection, which is a Highway with the gate fixed at 0.5 |
| `MoE` | `sum_i softmax(Wx)_i * expert_i(x)` -- N gates routing between N experts | Jacobs & Jordan, 1991, *Adaptive Mixtures of Local Experts* -- the dense version this notebook uses. Modern *sparse* MoE (Shazeer et al., 2017, *Outrageously Large Neural Networks*; the Switch Transformer, Fedus et al., 2021) routes each token to only a few of thousands of experts, which is the difference between this cell and a frontier language model, not a difference in the underlying idea |

Read top to bottom, each row is the row above with more capacity to choose from: one
gate choosing between a transform and nothing (`Gate`), one gate choosing between a
transform and the identity (`Highway`), N gates choosing between N transforms (`MoE`).

This lesson's electricity-demand series has real daily and weekly structure -- exactly
the kind of thing a router could plausibly use. Predict before you look at the table:
does giving the mechanism something real to route on change the verdict?


In [1]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components import Gate, Highway, MoE, NeuralNet

from australian_electricity import load_ausalia
from ett import window_index
from notebooktester import param

DAY = 48                # half-hourly steps per day
W, H = 4 * DAY, DAY      # four days of history, one day of horizon -- same window
                         # 04_probabilistic uses, so results sit next to that notebook's
HIDDEN = 64
STRIDE = 24
SEEDS = (0, 1, 2, 3, 4)

ausalia = load_ausalia()
mu = ausalia.values[ausalia.splits["train"]].mean()
sigma = ausalia.values[ausalia.splits["train"]].std()


def windows(split: str, stride: int):
    values = (ausalia.values[ausalia.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, stride)]
    return win[:, :W], win[:, W:]


xtr, ytr = windows("train", STRIDE)
xva, yva = windows("val", STRIDE * 2)
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}")


def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


def make_model(mechanism: str, hidden: int = HIDDEN, n_experts: int = 4) -> nn.Module:
    if mechanism == "plain":
        return NeuralNet(hidden_size=[W, hidden], out_size=H)
    if mechanism == "deeper":
        # capacity-matched control: one more Linear -> GELU than "plain", no gate at all.
        # Gate's own weight matrix costs exactly what this second layer costs at this width.
        return NeuralNet(hidden_size=[W, hidden, hidden], out_size=H)
    inp, act, out = nn.Linear(W, hidden), nn.GELU(), nn.Linear(hidden, H)
    if mechanism == "gate":
        mid = Gate(hidden_size=hidden)
    elif mechanism == "highway":
        mid = Highway(transform=NeuralNet(hidden, hidden), hidden_size=hidden)
    elif mechanism == "moe":
        mid = MoE(experts=[nn.Linear(hidden, hidden) for _ in range(n_experts)],
                  hidden_size=hidden, out_size=hidden)
    else:
        raise ValueError(mechanism)
    return nn.Sequential(inp, act, mid, out)


def run(mechanism: str, seed: int, epochs: int, lr: float = 1e-3, batch_size: int = 64) -> float:
    torch.manual_seed(seed)
    model = make_model(mechanism)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    loader = DataLoader(TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return mae(yva, model(xva))

train (8145, 192)   val (361, 192)


## 5.2 How many epochs is enough? Check across epoch counts *and* architectures

A single epoch count checked with 3 seeds is not enough evidence, and this notebook
found that out the hard way while it was being written: `gate` looked like a clear
winner over its capacity-matched control at one epoch count, and the advantage
vanished at another. The only fix is to check several epoch counts, with enough seeds
that a lucky run cannot decide the answer by itself.


In [2]:
EPOCHS_TO_CHECK = param((30, 60, 90), test=(1, 2))

for epochs in EPOCHS_TO_CHECK:
    for name in ("gate", "deeper"):
        scores = [run(name, seed, epochs) for seed in SEEDS]
        print(f"epochs={epochs:3d}   {name:7s} {np.mean(scores):.5f} +/- {np.std(scores):.5f}")
    print()

epochs= 30   gate    0.16739 +/- 0.00335


epochs= 30   deeper  0.16614 +/- 0.00321



epochs= 60   gate    0.16143 +/- 0.00666


epochs= 60   deeper  0.15982 +/- 0.00372



epochs= 90   gate    0.16026 +/- 0.01145


epochs= 90   deeper  0.15879 +/- 0.00699



`gate` and `deeper` -- its capacity-matched control, same parameter count, no gate
anywhere -- land inside each other's seed spread at every epoch count checked here.
That is the answer: pick a reasonably converged epoch count and stop looking for a
number where the gap opens up, because the checks above say there is not a stable one
to find. `EPOCHS = 90` below is not the best number in any of these rows; it is simply
past the region where training is still visibly moving.

## 5.3 The full family, at a settled epoch count


In [3]:
EPOCHS = param(90, test=2)

results: dict[str, dict] = {}
for name in ("plain", "deeper", "gate", "highway", "moe"):
    scores = [run(name, seed, EPOCHS) for seed in SEEDS]
    params = sum(p.numel() for p in make_model(name).parameters())
    results[name] = {"mae": float(np.mean(scores)), "std": float(np.std(scores)), "params": params}
    print(f"{name:8s} {results[name]['mae']:.5f} +/- {results[name]['std']:.5f}   {params:>7,} params")

plain    0.17381 +/- 0.01242    15,472 params


deeper   0.15879 +/- 0.00699    19,632 params


gate     0.16026 +/- 0.01145    19,632 params


highway  0.16155 +/- 0.00549    28,080 params


moe      0.16475 +/- 0.01009    32,372 params


## 5.4 The table

In [4]:
order = sorted(results, key=lambda k: results[k]["mae"])
print(f"{'model':10s} {'val MAE':>9} {'+/-':>7} {'params':>9}")
print("-" * 40)
for name in order:
    r = results[name]
    print(f"{name:10s} {r['mae']:9.5f} {r['std']:7.5f} {r['params']:9,}")

model        val MAE     +/-    params
----------------------------------------
deeper       0.15879 0.00699    19,632
gate         0.16026 0.01145    19,632
highway      0.16155 0.00549    28,080
moe          0.16475 0.01009    32,372
plain        0.17381 0.01242    15,472


`deeper` and `gate` are the two best rows, and they sit inside each other's seed
spread -- exactly the §5.2 finding, now with `plain`, `highway` and `moe` alongside
them for context. `highway` and `moe` cost 1.8x and 2.1x `plain`'s parameters and do
not beat it by a margin that clears their own seed noise either.

A series with real daily and weekly structure -- exactly the kind of thing a router
could plausibly use -- still does not make this family earn its parameters, once
checked properly. `09_composition_extra` (optional) measured the same shape of result
on a four-patch chunked window: a gate, a highway, or a mixture of experts is a real
bet that structure exists *and that this architecture can find it*, and the bet does
not pay just because the structure is there. `06_search` is where you stop checking
one hand-picked configuration at a time and let a search cover the space properly.

## What to take from this

- Gate, Highway and MoE are one idea -- a learned keep-or-replace decision -- at
  increasing resolution, and each one traces to a specific paper rather than floating
  free in a library's component list.
- The mechanism is general-purpose. It showed up here because this lesson had a window
  and a baseline ready, not because time series needs it more than any other modality.
- **A result checked at one epoch count is not checked.** §5.2 is in this notebook
  because the first version of it was wrong, and the fix was not a better number, it
  was more epoch counts and more seeds.
- Whether gating earns its extra parameters *on this window* is the number in the
  table above, not an opinion -- and "capacity-matched control" is doing the real work
  in that sentence, not "gate" or "highway" by name.

## Exercises

**E1 -- predict, then measure.** Before re-running this notebook, write down a ranking
you expect for the five trained models, and say whether you expect this series'
structure to change the §5.2 verdict from a null result to a real one. Check both.

**E2 -- turn the gate off.** Force `Gate`'s output to equal its input (patch the
forward pass, or initialise its linear layer to a large positive bias so the sigmoid
saturates near 1) and confirm the block's score converges toward `deeper`'s -- the
collapse argument from `1_pytorch_intro/02_nonlinearity` §2.1, for a gate instead of a
stacked linear layer.

**E3 -- more experts.** Re-run `make_model("moe")` with 2, 4, 8 and 16 experts, each
checked across the same epoch counts as §5.2. Does the router find anything to split
on, or does the score just track the parameter count?

**E4 -- a different window.** Swap in `pedestrian_counts` (loaded the same way
`03_decomposition_and_nbeats` and `06_search` both load it) and re-run the whole
notebook. Does the ranking change on a series with an even sharper daily cycle?
